<a href="https://colab.research.google.com/github/vidhu-psit/MachineLearning/blob/master/Recommendation_System/Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Groceries recommendation

In [1]:
# Download the groceries dataset from Google Drive
!gdown 14AGcLPV7t6r0mqkY2rPvSIcUgs8hoLWF

Downloading...
From: https://drive.google.com/uc?id=14AGcLPV7t6r0mqkY2rPvSIcUgs8hoLWF
To: /content/groceries-groceries.csv
100% 803k/803k [00:00<00:00, 39.4MB/s]


In [2]:
import pandas as pd
# Load the downloaded CSV file into a pandas DataFrame
df = pd.read_csv("groceries-groceries.csv")
# Display the first few rows of the DataFrame
df.head()

,Item(s),Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,...,Item 23,Item 24,Item 25,Item 26,Item 27,Item 28,Item 29,Item 30,Item 31,Item 32
0,4,citrus fruit,semi-finished bread,margarine,ready soups,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,tropical fruit,yogurt,coffee,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,whole milk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,pip fruit,yogurt,cream cheese,meat spreads,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,other vegetables,whole milk,condensed milk,long life bakery product,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Display the shape of the DataFrame (number of rows, number of columns)
df.shape

(9835, 33)

In [4]:
# Import necessary libraries for frequent pattern mining and association rule generation
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
# Import warnings and filter out DeprecationWarning from jupyter_client.session
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client.session')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Build list of list of each transaction to keep only bought items in each transaction

In [6]:
transaction =[]
# Iterate through each row of the DataFrame
for idx, row in df.iterrows():
  # Extract items from the row, skipping the first column (which is 'Item(s)')
  # and filter out NaN (Not a Number) values
  basket = [item for item in row[1:] if pd.notna(item)]
  # Append the list of items (basket) to the transactions list
  transaction.append(basket)

In [7]:
# Display the first transaction to inspect its contents
transaction[0]

['citrus fruit', 'semi-finished bread', 'margarine', 'ready soups']

In [8]:
# Get the total number of transactions processed
len(transaction)

9835

In [9]:
# Calculate the size (number of items) for each basket
basket_sizes = [len(t) for t in transaction]
# Convert basket sizes to a pandas Series and display descriptive statistics
pd.Series(basket_sizes).describe()

,0
count,9835.000000
mean,4.409456
std,3.589385
min,1.000000
25%,2.000000
50%,3.000000
75%,6.000000
max,32.000000


### Short Summary

The average basket size is approximately `4.4` items, with a range from `1` to `32` items. The majority of baskets (`50%`) contain `3` items or fewer.

<!-- This cell was empty and has been commented out. -->

In [10]:
# Initialize the TransactionEncoder to convert the list of transactions into a one-hot encoded DataFrame
te = TransactionEncoder()
# Fit the encoder to the transactions and transform them into a boolean matrix
item_matrix = te.fit_transform(transaction)
# Display the resulting item matrix
item_matrix

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False,  True, False],
       [False, False, False, ...,  True, False, False],
       ...,
       [False, False, False, ..., False,  True, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

In [11]:
# Convert the boolean item matrix into a pandas DataFrame, using the item names as column headers
df_items = pd.DataFrame(item_matrix, columns=te.columns_)
# Display the first few rows of the new DataFrame
df_items.head()

,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,baby food,bags,baking powder,bathroom cleaner,beef,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [12]:
# Calculate the support for each item by summing its occurrences and dividing by the total number of transactions
# Then, sort the items by support in descending order and display the top 15
df_items.sum().sort_values(ascending=False).head(15)/len(df_items)

,0
whole milk,0.255516
other vegetables,0.193493
rolls/buns,0.183935
soda,0.174377
yogurt,0.139502
bottled water,0.110524
root vegetables,0.108998
tropical fruit,0.104931
shopping bags,0.098526
sausage,0.093950


Milk appears in 25% of the transactions

# Apriori Algorithm

Minimum support 2%. Any item which has support less than 2 % will not be used
max len is 3 which means group of maximum 3 items are made.

## Use support to remove items which less than 2% support i.e. not bought more than 2% of transactions

In [13]:
# Apply the Apriori algorithm to find frequent itemsets
# min_support=0.02: itemsets must appear in at least 2% of transactions
# use_colnames=True: use actual item names in the output
# max_len=3: limit itemsets to a maximum of 3 items
# verbose=1: display progress messages
freq_sets = apriori(
    df_items,
    min_support=0.02,
    use_colnames=True,
    max_len=3,
    verbose=1
)

Processing 630 combinations | Sampling itemset size 3


**min_support=0.02**: This is a crucial parameter. It sets the minimum support threshold to 2%. This means that an itemset (a group of one or more items) must appear in at least 2% of all transactions to be considered 'frequent' and included in the results.


**max_len=3**: This limits the maximum number of items in an itemset to 3. So, the algorithm will find frequent individual items, pairs of items, and triplets of items, but no itemsets with 4 or more items.

In [14]:
# Display the frequent itemsets found by the Apriori algorithm
# This shows items with a support greater than 2%
freq_sets

,support,itemsets
0,0.033452,(UHT-milk)
1,0.052466,(beef)
2,0.033249,(berries)
3,0.026029,(beverages)
4,0.080529,(bottled beer)
...,...,...
117,0.032232,"(whipped/sour cream, whole milk)"
118,0.020742,"(yogurt, whipped/sour cream)"
119,0.056024,"(yogurt, whole milk)"
120,0.023183,"(other vegetables, root vegetables, whole milk)"


In [15]:

# Add a new column 'length' to the frequent itemsets DataFrame,
# indicating the number of items in each itemset
freq_sets["length"] = freq_sets.itemsets.str.len()
# Display the first few rows of the updated DataFrame
freq_sets.head()

,support,itemsets,length
0,0.033452,(UHT-milk),1
1,0.052466,(beef),1
2,0.033249,(berries),1
3,0.026029,(beverages),1
4,0.080529,(bottled beer),1


# Association rule mining

In [16]:
# Generate association rules from the frequent itemsets
# metric="confidence": evaluate rules based on confidence
# min_threshold=0.40: only consider rules with confidence of at least 40%
# Sort the rules by confidence in descending order
rules_conf = association_rules(
    freq_sets,
    metric="confidence",
    min_threshold=0.40
).sort_values("confidence", ascending=False)
# Display the generated association rules
rules_conf

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
14,"(yogurt, other vegetables)",(whole milk),0.043416,0.255516,0.022267,0.512881,2.007235,1.0,0.011174,1.528340,0.524577,0.080485,0.345695,0.300014
1,(butter),(whole milk),0.055414,0.255516,0.027555,0.497248,1.946053,1.0,0.013395,1.480817,0.514659,0.097237,0.324697,0.302543
2,(curd),(whole milk),0.053279,0.255516,0.026131,0.490458,1.919481,1.0,0.012517,1.461085,0.505984,0.092446,0.315577,0.296363
12,"(other vegetables, root vegetables)",(whole milk),0.047382,0.255516,0.023183,0.489270,1.914833,1.0,0.011076,1.457687,0.501524,0.082879,0.313982,0.289999
13,"(root vegetables, whole milk)",(other vegetables),0.048907,0.193493,0.023183,0.474012,2.449770,1.0,0.013719,1.533320,0.622230,0.105751,0.347821,0.296912
3,(domestic eggs),(whole milk),0.063447,0.255516,0.029995,0.472756,1.850203,1.0,0.013783,1.412030,0.490649,0.103800,0.291800,0.295073
10,(whipped/sour cream),(whole milk),0.071683,0.255516,0.032232,0.449645,1.759754,1.0,0.013916,1.352735,0.465077,0.109273,0.260757,0.287895
8,(root vegetables),(whole milk),0.108998,0.255516,0.048907,0.448694,1.756031,1.0,0.021056,1.350401,0.483202,0.154961,0.259479,0.320049
6,(root vegetables),(other vegetables),0.108998,0.193493,0.047382,0.434701,2.246605,1.0,0.026291,1.426693,0.622764,0.185731,0.299078,0.339789
4,(frozen vegetables),(whole milk),0.048094,0.255516,0.020437,0.424947,1.663094,1.0,0.008149,1.294636,0.418855,0.072172,0.227582,0.252466


<!-- This cell was empty and has been commented out. -->

Milk is suggested for every antecedent

## Using lift instead of confidence

In [17]:
# Generate association rules using 'lift' as the metric
# min_threshold=1.50: only consider rules with a lift of at least 1.50
# Sort the rules by lift in descending order
rules_lift = association_rules(
    freq_sets,
    metric="lift",
    min_threshold=1.50
).sort_values("lift", ascending=False)
# Display the generated association rules based on lift
rules_lift

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
68,(root vegetables),"(other vegetables, whole milk)",0.108998,0.074835,0.023183,0.212687,2.842082,1.0,0.015026,1.175091,0.727435,0.144304,0.149002,0.261235
65,"(other vegetables, whole milk)",(root vegetables),0.074835,0.108998,0.023183,0.309783,2.842082,1.0,0.015026,1.290900,0.700572,0.144304,0.225347,0.261235
40,(tropical fruit),(pip fruit),0.104931,0.075648,0.020437,0.194767,2.574648,1.0,0.012499,1.147931,0.683297,0.127619,0.128868,0.232464
41,(pip fruit),(tropical fruit),0.075648,0.104931,0.020437,0.270161,2.574648,1.0,0.012499,1.226392,0.661650,0.127619,0.184600,0.232464
66,"(root vegetables, whole milk)",(other vegetables),0.048907,0.193493,0.023183,0.474012,2.449770,1.0,0.013719,1.533320,0.622230,0.105751,0.347821,0.296912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44,(pork),(whole milk),0.057651,0.255516,0.022166,0.384480,1.504719,1.0,0.007435,1.209520,0.355945,0.076171,0.173226,0.235614
23,(fruit/vegetable juice),(other vegetables),0.072293,0.193493,0.021047,0.291139,1.504653,1.0,0.007059,1.137751,0.361531,0.085999,0.121073,0.199957
22,(other vegetables),(fruit/vegetable juice),0.193493,0.072293,0.021047,0.108776,1.504653,1.0,0.007059,1.040936,0.415861,0.085999,0.039326,0.199957
2,(bottled water),(soda),0.110524,0.174377,0.028978,0.262190,1.503577,1.0,0.009705,1.119017,0.376535,0.113230,0.106359,0.214185


Looking at antecedents and consequents. Now it is better recommendation.

In [18]:
# Generate candidate association rules with a minimum confidence of 0.40
candidate_rules = association_rules(
    freq_sets,
    metric="confidence",
    min_threshold=0.40
)
# Filter important rules based on support (>= 0.03) and lift (>= 1.60)
# Sort these important rules by confidence then by lift in descending order
important = candidate_rules[
    (candidate_rules["support"] >= 0.03) &
    (candidate_rules["lift"]    >= 1.60)
].sort_values(["confidence", "lift"], ascending=False)

# Select the top 5 important rules and reset their index
top5 = important.head(5).reset_index(drop=True)
# Display only the 'antecedents', 'consequents', 'support', 'confidence', and 'lift' columns for the top 5 rules
top5[["antecedents", "consequents",
      "support", "confidence", "lift"]]

,antecedents,consequents,support,confidence,lift
0,(whipped/sour cream),(whole milk),0.032232,0.449645,1.759754
1,(root vegetables),(whole milk),0.048907,0.448694,1.756031
2,(root vegetables),(other vegetables),0.047382,0.434701,2.246605


The output in the `top5` DataFrame represents the strongest association rules found, after filtering for a minimum support of 3% and a minimum lift of 1.6. These rules are sorted by confidence, then by lift, indicating the most reliable and interesting relationships.

Here's a quick interpretation of the top 3 rules:

1.  **{whipped/sour cream} -> {whole milk}**:
    *   **Confidence (44.96%)**: When a customer buys 'whipped/sour cream', there's a 44.96% chance they will also buy 'whole milk'.
    *   **Lift (1.76)**: Buying whipped/sour cream makes a customer 1.76 times more likely to buy whole milk than a random customer.

2.  **{root vegetables} -> {whole milk}**:
    *   **Confidence (44.87%)**: If a customer buys 'root vegetables', there's a 44.87% chance they will also buy 'whole milk'.
    *   **Lift (1.76)**: This implies they are 1.76 times more likely to buy whole milk.

3.  **{root vegetables} -> {other vegetables}**:
    *   **Confidence (43.47%)**: When 'root vegetables' are purchased, there's a 43.47% chance that 'other vegetables' are also bought.
    *   **Lift (2.25)**: This rule has the highest lift, meaning customers buying root vegetables are 2.25 times more likely to also buy other vegetables, highlighting a very strong positive association.

### Business Insights from Association Rules

The identified association rules provide valuable insights for sales and marketing strategies:

1.  **Cross-selling Opportunities (Dairy Products):**
    *   The rule ` {whipped/sour cream} -> {whole milk} ` (Confidence: 44.96%, Lift: 1.76) suggests a strong co-occurrence. Customers buying whipped/sour cream are significantly more likely to also buy whole milk.
    *   **Actionable Insight:** Place whipped/sour cream and whole milk closer together in stores. Offer promotions like 'Buy whipped/sour cream and get a discount on whole milk' or bundle them together in marketing campaigns.

2.  **Product Placement & Bundling (Vegetables & Dairy):**
    *   The rule ` {root vegetables} -> {whole milk} ` (Confidence: 44.87%, Lift: 1.76) indicates that customers purchasing root vegetables often also buy whole milk.
    *   **Actionable Insight:** Consider placing whole milk near the root vegetable section. Create meal-kit suggestions or recipes that feature both root vegetables and whole milk.

3.  **Category Management (Complementary Vegetables):**
    *   The rule ` {root vegetables} -> {other vegetables} ` (Confidence: 43.47%, Lift: 2.25) shows the strongest positive correlation (highest lift).
    *   **Actionable Insight:** This highlights a natural synergy. Ensure both 'root vegetables' and 'other vegetables' are well-stocked, visually appealing, and possibly cross-merchandised together. Promotions on one type of vegetable could boost sales of the other.

**Overall Strategy:**

*   **Optimized Store Layout:** Strategically placing associated items closer can encourage impulse purchases and increase basket size.
*   **Targeted Promotions:** Develop bundled offers or discounts for these strongly associated product pairs.
*   **Personalized Recommendations:** Use these rules to power recommendation engines for online shoppers or loyalty program members.
*   **Inventory Management:** Anticipate higher demand for associated items when one is selling well, ensuring adequate stock levels.

# 🎬 Indian Movies Recommendation System

In [19]:
# Import necessary libraries for data manipulation, numerical operations, plotting, machine learning, and warnings management.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [20]:
# Load the 'movies.csv' dataset into a pandas DataFrame
movies = pd.read_csv("movies.csv")
# Display the first few rows of the movies DataFrame
movies.head()

FileNotFoundError: [Errno 2] No such file or directory: 'movies.csv'

In [21]:
# Load the 'ratings.json' dataset into a pandas DataFrame, reading it line by line
ratings = pd.read_json('ratings.json', lines=True)
# Display the first few rows of the ratings DataFrame
rati<CUR_FLD_DEL>ngs.head()

FileNotFoundError: File ratings.json does not exist

In [22]:
# Transform nested ratings into a proper flat format
ratings_list = []
# Iterate over each row in the 'ratings' DataFrame
for _, row in ratings.iterrows():
  user_id = row['_id']
  # Check if 'rated' key exists and its value is a dictionary
  if 'rated' in row and isinstance(row['rated'], dict):
    # Iterate over movie_id and rating_value in the 'rated' dictionary
    for movie_id, rating_value in row['rated'].items():
      # Ensure it's not a 'submit' action and rating_value is a non-empty list
      if movie_id != 'submit' and isinstance(rating_value, list) and len(rating_value) > 0:
        # Convert the first element of rating_value to a float
        rating = float(rating_value[0])
        # Convert -1, 0, 1 ratings to a 1-5 scale for consistency
        if rating == -1:
            rating = 1
        elif rating == 0:
            rating = 3
        elif rating == 1:
            rating = 5
        # Append the transformed rating data to the list
        ratings_list.append({
            'userId': user_id,
            'movieId': movie_id,
            'rating': rating
        })

NameError: name 'ratings' is not defined

In [23]:
# Convert the list of processed ratings into a new pandas DataFrame
ratings = pd.DataFrame(ratings_list)

In [24]:
# This line is now redundant as the previous cell handles the rating transformation
# ratings.rating[ratings.rating == -1] = 0

In [ ]:
# Display the unique values in the 'rating' column to verify the transformation
ratings.rating.unique()

In [ ]:
# Load the 'users.csv' dataset into a pandas DataFrame
users = pd.read_csv('users.csv')
# Display the first few rows of the users DataFrame
users.head()

,_id,languages,job,state,dob,gender
0,11megha89,"[ ""Hindi"" ]",Student,Haryana,18-04-1989,Female
1,2ez4nimzi,"[ ""Hindi"" ]",Student,Delhi,16-06-2000,Male
2,9953547227,"[ ""Hindi"" ]",Student,Delhi,06-09-1998,Male
3,9958221803,"[ ""Hindi"" ]",Student,Delhi,09-09-1996,Male
4,ABCDEFGHI JKLM,"[ ""Hindi"" ]",Service,Delhi,26-01-1960,Male


In [ ]:
# Display the first few rows of the transformed ratings DataFrame
ratings.head()

,userId,movieId,rating
0,11megha89,tt0104561,5
1,11megha89,tt0323013,5
2,11megha89,tt2213054,3
3,11megha89,tt1447508,3
4,11megha89,tt4505006,3


# User - User based collaborative filtering recommendation

In [ ]:
# Create a user-item matrix where rows are users, columns are movies, and values are ratings.
# Fill NaN values (movies not rated by a user) with 0.
user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating',
    fill_value=0
)
# Display the first few rows of the user-item matrix
user_item_matrix.head()

movieId,tt0028626,tt0031829,tt0032836,tt0036077,tt0040067,tt0041123,tt0042329,tt0043307,tt0043908,tt0044360,...,tt5948916,tt5960464,tt5997928,tt6002986,tt6035564,tt6040012,tt6076366,tt6117534,tt6130166,tt6158988
userId,,,,,,,,,,,,,,,,,,,,,
11megha89,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9953547227,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABCDEFGHI JKLM,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ANAND,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Aakanksha,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


user

In [ ]:
# Display the shape of the user-item matrix (number of users, number of movies)
user_item_matrix.shape

(757, 1382)

In [ ]:
# Create a user similarity matrix using cosine similarity
# The index and columns of this matrix will both be user IDs
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
# Display the first few rows of the user similarity matrix
user_similarity_df.head()

userId,11megha89,9953547227,ABCDEFGHI JKLM,ANAND,Aakanksha,Aakash,Aakash22,Aashisha,Aayushi,Abhijeet Mishra,...,vishal kumar,vishuks,vishwanath maurya,vpsaini71@gmail.com,vrinda16279,yagami_kunal,yajur16121,yomojet,yusuf,zainab
userId,,,,,,,,,,,,,,,,,,,,,
11megha89,1.000000,0.046500,0.130051,0.0,0.122013,0.142481,0.028467,0.000000,0.149970,0.000000,...,0.020321,0.000000,0.081521,0.082317,0.053136,0.0,0.088163,0.079144,0.072412,0.050026
9953547227,0.046500,1.000000,0.000000,0.0,0.049543,0.028927,0.083482,0.000000,0.000000,0.077336,...,0.014898,0.000000,0.026895,0.000000,0.056097,0.0,0.000000,0.037979,0.038224,0.052813
ABCDEFGHI JKLM,0.130051,0.000000,1.000000,0.0,0.072169,0.000000,0.043778,0.000000,0.000000,0.000000,...,0.078125,0.094899,0.000000,0.000000,0.212459,0.0,0.000000,0.000000,0.000000,0.076932
ANAND,0.000000,0.000000,0.000000,1.0,0.000000,0.025826,0.096893,0.000000,0.058898,0.000000,...,0.000000,0.009694,0.000000,0.000000,0.000000,0.0,0.000000,0.005651,0.125129,0.062869
Aakanksha,0.122013,0.049543,0.072169,0.0,1.000000,0.035032,0.067401,0.019179,0.146472,0.056195,...,0.016238,0.032874,0.027143,0.000000,0.000000,0.0,0.000000,0.043440,0.023146,0.092387


In [ ]:
# Display the shape of the user similarity matrix
user_similarity_df.shape

(757, 757)

# top 10 similar users to "11megha89"

In [ ]:
# Identify the top 10 most similar users to the specified user_id, excluding the user himself
user_id = '11megha89'
similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:11]
# Display the list of similar users and their similarity scores
similar_users

,11megha89
userId,
richa239,0.325128
rachna10,0.312819
seemasood71,0.298184
rahul16074,0.282954
Shweta Rana,0.281569
singhjaibir68,0.280275
Zoha,0.256873
dr_rajive,0.248164
Ashish_16,0.237405


Similar user IDs

In [ ]:
# Display the index (user IDs) of the similar users
similar_users.index

Index(['richa239', 'rachna10', 'seemasood71', 'rahul16074', 'Shweta Rana',
       'singhjaibir68', 'Zoha', 'dr_rajive', 'Ashish_16', 'madhav16159'],
      dtype='object', name='userId')

In [ ]:
# Filter out rows for these similar users from the user_item_matrix
# to get the ratings given by these similar users
similar_users_ratings = user_item_matrix.loc[similar_users.index]
# Display the ratings of similar users
similar_users_ratings

movieId,tt0028626,tt0031829,tt0032836,tt0036077,tt0040067,tt0041123,tt0042329,tt0043307,tt0043908,tt0044360,...,tt5948916,tt5960464,tt5997928,tt6002986,tt6035564,tt6040012,tt6076366,tt6117534,tt6130166,tt6158988
userId,,,,,,,,,,,,,,,,,,,,,
richa239,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
rachna10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
seemasood71,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
rahul16074,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Shweta Rana,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
singhjaibir68,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zoha,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dr_rajive,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Ashish_16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Calculate the weighted average of ratings for movies from similar users
# Each rating is weighted by the similarity score of the user, then normalized by the sum of similarities
weighted_ratings = similar_users_ratings.T.dot(similar_users.values) / similar_users.sum()
# Display the calculated weighted ratings
weighted_ratings

,0
movieId,
tt0028626,0.000000
tt0031829,0.449457
tt0032836,0.000000
tt0036077,0.000000
tt0040067,0.085971
...,...
tt6040012,0.000000
tt6076366,0.000000
tt6117534,0.000000


In [ ]:
# Get the ratings of the target user ('11megha89')
user_rated = user_item_matrix.loc[user_id]
# Filter out movies that the target user has already rated (where user_rated is not 0)
# This leaves only movies that the user has not yet seen or rated
recommendations = weighted_ratings[user_rated == 0]
# Display the recommendations (movies not yet rated by the user)
recommendations

,0
movieId,
tt0028626,0.000000
tt0031829,0.449457
tt0032836,0.000000
tt0036077,0.000000
tt0040067,0.085971
...,...
tt6040012,0.000000
tt6076366,0.000000
tt6117534,0.000000


In [ ]:
# Filter the top 10 recommendations by sorting the weighted ratings in descending order
top_recommendations = recommendations.sort_values(ascending=False).head(10)

In [ ]:
# Display the top 10 movie recommendations with their predicted ratings
top_recommendations

,0
movieId,
tt0112870,1.962347
tt0066070,1.517743
tt0499375,1.050006
tt0077797,1.047661
tt4434004,0.942436
tt1926313,0.817437
tt0072777,0.797963
tt0242519,0.674564
tt0234225,0.566555


In [ ]:
# Create a DataFrame from the top recommendations
results = pd.DataFrame({
            'movieId': top_recommendations.index,
            'predicted_rating': top_recommendations.values
        })

In [ ]:
# Display the results DataFrame, showing movie IDs and predicted ratings
results

,movieId,predicted_rating
0,tt0112870,1.962347
1,tt0066070,1.517743
2,tt0499375,1.050006
3,tt0077797,1.047661
4,tt4434004,0.942436
5,tt1926313,0.817437
6,tt0072777,0.797963
7,tt0242519,0.674564
8,tt0234225,0.566555
9,tt0050132,0.566555


In [ ]:
# Merge the 'results' DataFrame with the 'movies' DataFrame to add movie names
# It joins on 'movieId' from 'results' and 'movie_id' from 'movies'
results = pd.merge(results, movies[['movie_id', 'name']], left_on='movieId', right_on='movie_id', how='left')
# Drop the redundant 'movie_id' column
results = results.drop(columns=['movie_id'])
# Rename the 'name' column to 'movie_name' for clarity
results = results.rename(columns={'name': 'movie_name'})
# Display the final results with movie names
results

,movieId,predicted_rating,movie_name,movie_name
0,tt0112870,1.962347,Dilwale Dulhania Le Jayenge,Dilwale Dulhania Le Jayenge
1,tt0066070,1.517743,Mera Naam Joker,Mera Naam Joker
2,tt0499375,1.050006,Guru,Guru
3,tt0077797,1.047661,Khatta Meetha,Khatta Meetha
4,tt4434004,0.942436,Udta Punjab,Udta Punjab
5,tt1926313,0.817437,Pyaar Ka Punchnama,Pyaar Ka Punchnama
6,tt0072777,0.797963,Chhoti Si Baat,Chhoti Si Baat
7,tt0242519,0.674564,Hera Pheri,Hera Pheri
8,tt0234225,0.566555,Meet Mere Man Ke,Meet Mere Man Ke
9,tt0050132,0.566555,Apradhi Kaun?,Apradhi Kaun?


In [ ]:
def get_user_recommendations(user_id, user_item_matrix, user_similarity_df, movies, num_recommendations=10):
  # Identify top similar user by sorting cosine similarity values for the given user_id
  # [1:11] is used to exclude the user himself and get the top 10 similar users.
  similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:11]

  # Get ratings of movies from these similar users by filtering the user_item_matrix
  similar_users_ratings = user_item_matrix.loc[similar_users.index]

  # Calculate weighted ratings for movies. This is done by taking the dot product of
  # the transposed similar_users_ratings with the similarity scores of similar_users,
  # and then dividing by the sum of similarity scores to normalize.
  weighted_ratings = similar_users_ratings.T.dot(similar_users.values) / similar_users.sum()

  # Filter out movies which the target user has already rated
  # First, get the target user's ratings from the user-item matrix.
  user_rated = user_item_matrix.loc[user_id]
  # Then, select only those movies from weighted_ratings that the user has not rated (rating is 0).
  recommendations = weighted_ratings[user_rated == 0]

  # Get top recommendations by sorting the unrated movies by their weighted ratings in descending order
  # and taking the top 'num_recommendations'.
  top_recommendations = recommendations.sort_values(ascending=False).head(num_recommendations)

  # Create a DataFrame for the results, including the movie ID and predicted rating.
  results = pd.DataFrame({
              'movieId': top_recommendations.index,
              'predicted_rating': top_recommendations.values
          })

  # Add movie names for better readability by merging with the 'movies' DataFrame.
  results = pd.merge(results, movies[['movie_id', 'name']], left_on='movieId', right_on='movie_id', how='left')
  # Drop the duplicate 'movie_id' column that resulted from the merge.
  results = results.drop(columns=['movie_id'])
  # Rename the 'name' column to 'movie_name'.
  results = results.rename(columns={'name': 'movie_name'})

  return results

# Example usage:
# user_id = '11megha89'
# recommendations_df = get_user_recommendations(user_id, user_item_matrix, user_similarity_df, movies)
# print(recommendations_df)

In [ ]:
import random

# Get a list of all user IDs from the user-item matrix index
all_user_ids = user_item_matrix.index.tolist()

# Select a random user ID from the list
random_user_id = random.choice(all_user_ids)

print(f"Getting recommendations for random user: {random_user_id}")

# Get recommendations for the randomly selected user using the defined function
random_user_recommendations = get_user_recommendations(random_user_id, user_item_matrix, user_similarity_df, movies)
# Display the recommendations for the random user
random_user_recommendations

Getting recommendations for random user: seema


,movieId,predicted_rating,movie_name
0,tt0453729,1.493898,Iqbal
1,tt0488414,1.485411,Omkara
2,tt2082197,1.269856,Barfi!
3,tt5713232,1.264525,Madaari
4,tt1954470,1.144546,Gangs of Wasseypur
5,tt0067403,1.100347,Maryada
6,tt0242519,1.083054,Hera Pheri
7,tt0214841,1.083054,Karz
8,tt0362038,1.067379,Pratyaghat
9,tt0292490,1.065555,Dil Chahta Hai


# Item - Item based recommendation

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Create an Item-User Matrix (transpose of user_item_matrix)
# The user_item_matrix has users as rows and movies as columns.
# For item-item similarity, we need movies as rows and users as columns.
item_user_matrix = user_item_matrix.T
print("Item-User Matrix Head:")
print(item_user_matrix.head())

# 2. Calculate Item-Item Similarity using Cosine Similarity
# This matrix will show how similar each movie is to every other movie.
item_similarity = cosine_similarity(item_user_matrix)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)
print("\nItem-Item Similarity Matrix Head:")
print(item_similarity_df.head())

def get_item_recommendations(
    movie_id: str,
    user_id: str,
    item_user_matrix: pd.DataFrame,
    item_similarity_df: pd.DataFrame,
    movies_df: pd.DataFrame,
    num_recommendations: int = 10
) -> pd.DataFrame:
    """
    Generates item-item based recommendations for a given user based on a specific movie.

    Args:
        movie_id (str): The ID of the movie to base recommendations on.
        user_id (str): The ID of the user for whom to generate recommendations.
        item_user_matrix (pd.DataFrame): Item-user matrix (movies as rows, users as columns).
        item_similarity_df (pd.DataFrame): DataFrame of item-item cosine similarities.
        movies_df (pd.DataFrame): DataFrame containing movie metadata (movie_id, name).
        num_recommendations (int): The number of recommendations to return.

    Returns:
        pd.DataFrame: A DataFrame with recommended movie IDs, names, and similarity scores.
    """

    if movie_id not in item_similarity_df.index:
        print(f"Error: Movie ID '{movie_id}' not found in the similarity matrix.")
        return pd.DataFrame(columns=['movieId', 'movie_name', 'similarity_score'])
    if user_id not in item_user_matrix.columns:
        print(f"Error: User ID '{user_id}' not found in the item-user matrix.")
        return pd.DataFrame(columns=['movieId', 'movie_name', 'similarity_score'])

    # Get similar movies for the given movie_id, excluding the movie itself
    similar_movies_scores = item_similarity_df[movie_id].sort_values(ascending=False)
    similar_movies_scores = similar_movies_scores.drop(movie_id, errors='ignore')

    # Get movies already rated by the target user
    user_rated_movies = item_user_matrix[user_id]
    rated_by_user_ids = user_rated_movies[user_rated_movies > 0].index.tolist()

    # Filter out movies already rated by the user
    unrated_similar_movies = similar_movies_scores.drop(rated_by_user_ids, errors='ignore')

    # Get the top N recommendations
    top_recommendations = unrated_similar_movies.head(num_recommendations)

    # Create a DataFrame for results
    results = pd.DataFrame({
        'movieId': top_recommendations.index,
        'similarity_score': top_recommendations.values
    })

    # Add movie names for better readability
    results = pd.merge(results, movies_df[['movie_id', 'name']], left_on='movieId', right_on='movie_id', how='left')
    results = results.drop(columns=['movie_id'])
    results = results.rename(columns={'name': 'movie_name'})

    return results

# Example Usage:
# Let's pick a random movie from the existing movie_id in the item_user_matrix
# And a user_id that exists.
example_movie_id = item_user_matrix.index[10] # Picking an arbitrary movie_id
example_user_id = user_item_matrix.index[0] # Picking '11megha89'

print(f"\nGetting item-item recommendations for user '{example_user_id}' based on movie '{example_movie_id}':")
item_recommendations_df = get_item_recommendations(
    movie_id=example_movie_id,
    user_id=example_user_id,
    item_user_matrix=item_user_matrix,
    item_similarity_df=item_similarity_df,
    movies_df=movies
)

if not item_recommendations_df.empty:
    print(item_recommendations_df)
else:
    print("No recommendations found or an error occurred.")


Item-User Matrix Head:
userId     11megha89  9953547227   ABCDEFGHI JKLM  ANAND  Aakanksha  Aakash  \
movieId                                                                       
tt0028626        0.0          0.0             0.0    0.0        0.0     0.0   
tt0031829        0.0          0.0             0.0    0.0        0.0     0.0   
tt0032836        0.0          0.0             0.0    0.0        0.0     0.0   
tt0036077        0.0          0.0             0.0    0.0        0.0     0.0   
tt0040067        0.0          0.0             0.0    0.0        0.0     0.0   

userId     Aakash22  Aashisha  Aayushi  Abhijeet Mishra   ...  vishal kumar  \
movieId                                                   ...                 
tt0028626       0.0       0.0      0.0               0.0  ...           0.0   
tt0031829       0.0       0.0      0.0               0.0  ...           0.0   
tt0032836       0.0       0.0      0.0               0.0  ...           0.0   
tt0036077       0.0       0.

In [ ]:
# This cell was empty and has been commented out. There is no code to execute or explain here.